# SpaceX Falcon 9 Data Collection - Web Scraping

**IBM Applied Data Science Capstone**  
Repository owner: `phuonganhdt220391`

This notebook demonstrates a reproducible web-scraping workflow. To keep the
saved notebook stable when Wikipedia changes its page structure, the default
execution uses the course snapshot `Spacex.csv`. Set `USE_LIVE_WEB = True`
to attempt a fresh scrape with `pandas.read_html`.

## Objectives

1. Retrieve launch tables from the Falcon 9 launch-list page.
2. Identify tables containing launch records.
3. Normalize columns into a clean tabular dataset.
4. Export the result as `spacex_web_scraped.csv`.

In [1]:
from pathlib import Path
import re
import pandas as pd

pd.set_option("display.max_columns", 20)
DATA_DIR = Path(".")
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
USE_LIVE_WEB = False

In [2]:
def flatten_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Convert MultiIndex/verbose HTML table headers to simple strings."""
    clean = frame.copy()
    if isinstance(clean.columns, pd.MultiIndex):
        clean.columns = [
            " ".join(str(part) for part in col if str(part) != "nan").strip()
            for col in clean.columns
        ]
    clean.columns = [re.sub(r"\[.*?\]", "", str(c)).strip() for c in clean.columns]
    return clean


def scrape_launch_tables(url: str) -> pd.DataFrame:
    """Read HTML tables and keep tables that look like launch records."""
    tables = [flatten_columns(t) for t in pd.read_html(url)]
    candidates = []
    for table in tables:
        header = " | ".join(table.columns).lower()
        if "flight" in header and "launch" in header and len(table) >= 3:
            candidates.append(table)
    if not candidates:
        raise ValueError("No launch-record tables were detected.")
    return pd.concat(candidates, ignore_index=True, sort=False)

In [3]:
if USE_LIVE_WEB:
    try:
        raw = scrape_launch_tables(WIKI_URL)
        source_used = "Live Wikipedia tables"
    except Exception as exc:
        print(f"Live scrape unavailable ({type(exc).__name__}: {exc}). Using course snapshot.")
        raw = pd.read_csv(DATA_DIR / "Spacex.csv")
        source_used = "IBM Skills Network course snapshot"
else:
    raw = pd.read_csv(DATA_DIR / "Spacex.csv")
    source_used = "IBM Skills Network course snapshot"

print("Source:", source_used)
print("Rows, columns:", raw.shape)
raw.head()

Source: IBM Skills Network course snapshot
Rows, columns: (101, 10)


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [4]:
# The course snapshot is already normalized. For a live scrape, retain the
# detected launch rows and standardize whitespace before export.
scraped = raw.copy()
scraped.columns = [str(c).strip().replace(" ", "_") for c in scraped.columns]
for col in scraped.select_dtypes(include="object").columns:
    scraped[col] = scraped[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

output_path = DATA_DIR / "spacex_web_scraped.csv"
scraped.to_csv(output_path, index=False)
print(f"Saved {len(scraped)} rows to {output_path}")
scraped.head(10)

Saved 101 rows to spacex_web_scraped.csv


,Date,Time_(UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt
5,2013-09-29,16:00:00,F9 v1.1 B1003,VAFB SLC-4E,CASSIOPE,500,Polar LEO,MDA,Success,Uncontrolled (ocean)
6,2013-12-03,22:41:00,F9 v1.1,CCAFS LC-40,SES-8,3170,GTO,SES,Success,No attempt
7,2014-01-06,22:06:00,F9 v1.1,CCAFS LC-40,Thaicom 6,3325,GTO,Thaicom,Success,No attempt
8,2014-04-18,19:25:00,F9 v1.1,CCAFS LC-40,SpaceX CRS-3,2296,LEO (ISS),NASA (CRS),Success,Controlled (ocean)
9,2014-07-14,15:15:00,F9 v1.1,CCAFS LC-40,OG2 Mission 1 6 Orbcomm-OG2 satellites,1316,LEO,Orbcomm,Success,Controlled (ocean)


## Result

The cleaned table is stored in `spacex_web_scraped.csv`. The saved snapshot
makes the repository readable and reproducible even when the live web page
changes after the course was published.